# Challenge 2: entrenamiento y selección del modelo

En esta etapa debes construir pipelines reproducibles, comparar modelos y seleccionar un campeón sin utilizar el conjunto de test.

**Entradas:** `train.csv` y `data_contract.json`  
**Salidas:** `champion_model.joblib`, `training_metadata.json` y `cv_results.csv`

## Reglas

- El preprocesamiento debe estar dentro de cada `Pipeline`.
- La selección debe hacerse con validación cruzada.
- No cargues `test.csv` en este notebook.

In [1]:
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_validate
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

In [2]:
from pathlib import Path

PROJECT_DIR = Path.cwd()
RAW_DATA_DIR = PROJECT_DIR / "data" 
PROCESSED_DATA_DIR = PROJECT_DIR / "data" / "processed"
ARTIFACTS_DIR = PROJECT_DIR / "artifacts"
REPORTS_DIR = PROJECT_DIR / "reports"

for directory in (PROCESSED_DATA_DIR, ARTIFACTS_DIR, REPORTS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42

## 1. Carga de artefactos

In [3]:
train_df = pd.read_csv(PROCESSED_DATA_DIR / "train.csv")
contract = json.loads((ARTIFACTS_DIR / "data_contract.json").read_text(encoding="utf-8"))

X_train = train_df[contract["features"]]
y_train = train_df[contract["target"]]

print(X_train.shape, y_train.shape)

(142, 13) (142,)


## 2. Pipeline de procesamiento

In [4]:
feature_columns = contract["features"]

preprocessor_tree = ColumnTransformer(
    transformers=[
        ("numeric", SimpleImputer(strategy="median"), feature_columns)
    ],
    remainder="drop"
)

preprocessor_scaled = ColumnTransformer(
    transformers=[
        ("numeric", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), feature_columns)
    ],
    remainder="drop"
)

**Pregunta:** ¿por qué el `StandardScaler` debe estar dentro del pipeline y no ajustarse antes de la validación cruzada?

Para que, en cada fold, se ajuste únicamente con los datos de entrenamiento de ese fold. Luego transforma el fold de validación usando esas mismas estadísticas. Si se ajusta antes de la validación cruzada, la información de los folds de validación influye en el escalado y produce fuga de datos, lo que puede sobreestimar el rendimiento del modelo.

## 3. Modelos candidatos

In [5]:
models = {
    "DecisionTree": Pipeline([
        ("preprocessor", clone(preprocessor_tree)),
        ("classifier", DecisionTreeClassifier(random_state=RANDOM_STATE))
    ]),
    "LogisticRegression": Pipeline([
        ("preprocessor", clone(preprocessor_scaled)),
        ("classifier", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
    ]),
    "KNN": Pipeline([
        ("preprocessor", clone(preprocessor_scaled)),
        ("classifier", KNeighborsClassifier())
    ])
}

## 4. Comparación inicial con validación cruzada

In [6]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

comparison_rows = []
for model_name, model in models.items():
    scores = cross_validate(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring="accuracy",
        return_train_score=True
    )
    comparison_rows.append({
        "model": model_name,
        "train_accuracy_mean": scores["train_score"].mean(),
        "validation_accuracy_mean": scores["test_score"].mean(),
        "validation_accuracy_std": scores["test_score"].std()
    })

initial_comparison = pd.DataFrame(comparison_rows).sort_values(
    "validation_accuracy_mean", ascending=False
).reset_index(drop=True)

initial_comparison

,model,train_accuracy_mean,validation_accuracy_mean,validation_accuracy_std
0,LogisticRegression,1.0000,0.979064,0.027713
1,KNN,0.9771,0.950739,0.035274
2,DecisionTree,1.0000,0.902217,0.058617


## 5. Optimización de hiperparámetros

In [7]:
param_grids = {
    "DecisionTree": {
        "classifier__max_depth": [None, 3, 5],
        "classifier__min_samples_leaf": [1, 2]
    },
    "LogisticRegression": {
        "classifier__C": [0.1, 1, 10]
    },
    "KNN": {
        "classifier__n_neighbors": [3, 5, 7],
        "classifier__weights": ["uniform", "distance"]
    }
}

searches = {}
tuned_rows = []

for model_name, model in models.items():
    search = GridSearchCV(
        estimator=model,
        param_grid=param_grids[model_name],
        scoring="accuracy",
        cv=cv,
        n_jobs=-1,
        refit=True
    )
    search.fit(X_train, y_train)
    searches[model_name] = search

    best_idx = search.best_index_
    tuned_rows.append({
        "model": model_name,
        "best_accuracy": search.best_score_,
        "validation_accuracy_std": search.cv_results_["std_test_score"][best_idx],
        "best_params": search.best_params_
    })

tuned_comparison = (
    pd.DataFrame(tuned_rows)
    .sort_values("best_accuracy", ascending=False)
    .reset_index(drop=True)
)

tuned_comparison

,model,best_accuracy,validation_accuracy_std,best_params
0,LogisticRegression,0.979064,0.027713,{'classifier__C': 1}
1,KNN,0.957882,0.040520,"{'classifier__n_neighbors': 7, 'classifier__we..."
2,DecisionTree,0.902463,0.076225,"{'classifier__max_depth': None, 'classifier__m..."


## 6. Selección y persistencia del campeón

In [8]:
champion_name = max(searches, key=lambda name: searches[name].best_score_)
champion = searches[champion_name].best_estimator_

joblib.dump(champion, ARTIFACTS_DIR / "champion_model.joblib")

best_search = searches[champion_name]
training_metadata = {
    "champion_model": champion_name,
    "selection_metric": "accuracy",
    "best_cv_accuracy": float(best_search.best_score_),
    "cv_accuracy_std": float(
        best_search.cv_results_["std_test_score"][best_search.best_index_]
    ),
    "best_params": best_search.best_params_,
    "cv_folds": cv.get_n_splits(),
    "random_state": RANDOM_STATE,
    "features": feature_columns,
    "target": contract["target"],
}
(ARTIFACTS_DIR / "training_metadata.json").write_text(
    json.dumps(training_metadata, indent=2, default=str),
    encoding="utf-8",
)

cv_results = pd.concat(
    [
        pd.DataFrame(search.cv_results_).assign(model=name)
        for name, search in searches.items()
    ],
    ignore_index=True,
)
cv_results.to_csv(REPORTS_DIR / "cv_results.csv", index=False)

print(f"Campeón: {champion_name} (accuracy CV: {best_search.best_score_:.4f})")

Campeón: LogisticRegression (accuracy CV: 0.9791)


In [9]:
assert (ARTIFACTS_DIR / "champion_model.joblib").exists()
assert (ARTIFACTS_DIR / "training_metadata.json").exists()
assert (REPORTS_DIR / "cv_results.csv").exists()
print("Challenge 2 completado.")

Challenge 2 completado.


**Justificación de la selección:**

> Explica la decisión considerando desempeño, variabilidad, sobreajuste, complejidad e interpretabilidad.

Se eligió LogisticRegression como modelo campeón porque obtuvo el mejor rendimiento con validación cruzada, accuracy  de 0.9791, superior a KNN (0.9579) y a DecisionTree (0.9025). Además, presentó la menor variabilidad entre folds (std = 0.0277), lo que indica resultados más estables y menos sensibles a la partición de los datos. 

El aumento de diferencia entre entrenamiento y validación en KNN y en el árbol sugiere más sobreajuste y peor generalización. En términos de complejidad, la regresión logística es un modelo más simple y regularizado, con menos riesgo de memorizar ruido, el árbol, aunque interpretable, tiene mayor variabilidad y peor estabilidad. KNN depende de la distancia local y requiere más ajuste de hiperparámetros. Finalmente, la regresión logística es también más interpretable, ya que permite analizar la influencia de cada característica mediante los coeficientes, algo muy útil en un problema de clasificación con variables continuas como este.